In [1]:
import pandas as pd
import numpy as np
import os

from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

# -----------------------------
# SNV transformation
# -----------------------------
def apply_snv(X):
    # Standard Normal Variate: corrects multiplicative scatter and baseline
    mean = np.mean(X, axis=1, keepdims=True)
    std = np.std(X, axis=1, keepdims=True)
    return (X - mean) / std

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("../data/train.csv", encoding="cp932")
test  = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# -----------------------------
# Define spectral columns
# -----------------------------
spectral_cols = [
    c for c in train.columns
    if c not in ["sample number", "species number", "樹種", "含水率"]
]

X = train[spectral_cols].values
y = train["含水率"].values
X_test = test[spectral_cols].values

print("Number of spectral features:", X.shape[1])

# -----------------------------
# Preprocessing: SNV + SG 2nd derivative
# -----------------------------
print("Applying SNV transformation...")
X_snv = apply_snv(X)
X_test_snv = apply_snv(X_test)

print("Applying Savitzky-Golay second derivative...")
X_sg2 = savgol_filter(X_snv, window_length=21, polyorder=2, deriv=2, axis=1)
X_test_sg2 = savgol_filter(X_test_snv, window_length=21, polyorder=2, deriv=2, axis=1)

print("Preprocessing complete: SNV + SG 2nd derivative")

# -----------------------------
# Scale features
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sg2)
X_test_scaled = scaler.transform(X_test_sg2)

# -----------------------------
# Train PLS Regression
# -----------------------------
pls = PLSRegression(n_components=12)
pls.fit(X_scaled, y)

# Predict test
test_preds = pls.predict(X_test_scaled).flatten()

print("Sample predictions:", test_preds[:10])

# -----------------------------
# Save submission
# -----------------------------
os.makedirs("../submissions", exist_ok=True)
submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

output_path = "../submissions/exp_nature_snv_sg2_pls_20260323.csv"
submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved:", output_path)
print(pd.read_csv(output_path, header=None).head())

Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of spectral features: 1555
Applying SNV transformation...
Applying Savitzky-Golay second derivative...
Preprocessing complete: SNV + SG 2nd derivative
Sample predictions: [165.25186705 156.86052123 159.44156228 154.67447256 141.78500422
 150.3893753  138.36222462 143.49016215 138.09700433 143.87278156]

Submission saved: ../submissions/exp_nature_snv_sg2_pls_20260323.csv
    0           1
0  95  165.251867
1  96  156.860521
2  97  159.441562
3  98  154.674473
4  99  141.785004
